In [ ]:
"""
Objective: Execute Rolling-Window VAR models. Generates Subsection 4.3 outputs 
(48-month baseline window with crisis anchor regimes) and Appendix A.2 multi-window 
robustness panels (36 to 84 months).
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})

# ==========================================
# 0. GENERAL PARAMETERS
# ==========================================
CSV_FILE = '07_clean_rolling_var_fred.csv'
CHOLESKY_ORDER = ['TED_Spread', 'Unemployment', 'Fed_Funds'] 
WINDOW_LIST = [36, 48, 60, 72, 84]
BASELINE_WINDOW = 48
VAR_LAG = 1
IRF_HORIZON = 12       
FEVD_HORIZON = 6       
FORECAST_HORIZON = 3   
MC_REPLICATIONS = 500      
SIGNIFICANCE = 0.10            
SEED = 42

IRF_REGIMES = {
    'Pre-Crisis (2006)': '2006-12-01',
    'Subprime (2007)': '2007-12-01',
    'Lehman Bros. (2008)': '2008-12-01',
}
COLORS_REGIMES = {
    'Pre-Crisis (2006)': '#1f77b4',
    'Subprime (2007)': '#ff7f0e',
    'Lehman Bros. (2008)': '#d62728',
}

# ==========================================
# 1. MACROECONOMIC VECTOR PREPARATION
# ==========================================
if not os.path.exists(CSV_FILE):
    raise SystemExit(f"[ERROR] File '{CSV_FILE}' not found. Run previous steps first.")

df = pd.read_csv(CSV_FILE, parse_dates=['Date'], index_col='Date')[CHOLESKY_ORDER]
df = df.diff().dropna() 

idx_ted = CHOLESKY_ORDER.index('TED_Spread')
idx_un = CHOLESKY_ORDER.index('Unemployment')
idx_fed = CHOLESKY_ORDER.index('Fed_Funds')

# ==========================================
# 2. SUBSECTION 4.3 OUTPUTS (48-Month Baseline Window)
# ==========================================
def generate_section_43_outputs():
    print("[*] Generating Subsection 4.3 figures (48-month baseline window)...")
    
    # A. IRF Panel by Regimes (Figure 7 equivalent)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    months = np.arange(IRF_HORIZON + 1)
    
    for idx, (name, end_date_str) in enumerate(IRF_REGIMES.items()):
        end_date = pd.to_datetime(end_date_str)
        start_date = end_date - pd.DateOffset(months=BASELINE_WINDOW)
        window_df = df.loc[start_date:end_date]
        
        res = VAR(window_df).fit(VAR_LAG)
        irf = res.irf(IRF_HORIZON)
        response = irf.orth_irfs[:, idx_fed, idx_ted]
        lower, upper = irf.errband_mc(orth=True, repl=MC_REPLICATIONS, signif=SIGNIFICANCE, seed=SEED)
        
        ax = axes[idx]
        ax.plot(months, response, color=COLORS_REGIMES[name], linewidth=2.5)
        ax.fill_between(months, lower[:, idx_fed, idx_ted], upper[:, idx_fed, idx_ted], color=COLORS_REGIMES[name], alpha=0.15)
        ax.axhline(0, color='black', linewidth=1, linestyle='--')
        ax.set_title(name, fontsize=11, fontweight='bold')
        ax.set_xlabel('Months after shock', fontsize=10, fontweight='bold')
        if idx == 0:
            ax.set_ylabel('Fed Response (Change)', fontsize=10, fontweight='bold')
            
    plt.tight_layout()
    # CORREÇÃO: Nome do arquivo em inglês
    plt.savefig('09b_irf_regimes_panel.png', dpi=300, bbox_inches='tight')
    plt.close()

    # B. Rolling-FEVD for 48-month window (Figure 8 equivalent)
    dates, rows = [], []
    for i in range(BASELINE_WINDOW, len(df)):
        window_df = df.iloc[i - BASELINE_WINDOW: i]
        try:
            fevd = VAR(window_df).fit(VAR_LAG).fevd(FEVD_HORIZON)
            d = fevd.decomp[idx_fed, -1, :]
            dates.append(df.index[i - 1])
            rows.append([d[idx_fed], d[idx_ted], d[idx_un]])
        except:
            continue
            
    df_fevd = pd.DataFrame(np.array(rows) * 100, index=dates,
                           columns=['Own Shocks (Fed)', 'TED Spread', 'Unemployment'])
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.stackplot(df_fevd.index, df_fevd['Own Shocks (Fed)'], df_fevd['TED Spread'], df_fevd['Unemployment'],
                 labels=['Own Shocks (Fed)', 'Liquidity Risk (TED Spread)', 'Real Economy (Unemployment)'],
                 colors=['#2ca02c', '#d62728', '#1f77b4'], alpha=0.85)
                 
    ax.axvline(pd.to_datetime('2007-08-01'), color='orange', linestyle='--', linewidth=1.5, label='Subprime Start')
    ax.axvline(pd.to_datetime('2008-09-15'), color='black', linestyle=':', linewidth=2.0, label='Lehman Bros. Collapse')
    ax.set_ylabel('Explained Proportion (%)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Year', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=9)
    
    plt.tight_layout()
    # CORREÇÃO: Nome do arquivo em inglês
    plt.savefig('09b_stacked_fevd_panel.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("[SUCCESS] Section 4.3 figures successfully generated.")

# ==========================================
# 3. APPENDIX A.2 OUTPUTS (Multi-window Robustness Panels)
# ==========================================
def generate_appendix_robustness_panels():
    print("[*] Generating Appendix A.2 multi-window robustness panels...")
    
    # A. Multi-window IRF Panel (36 to 84m)
    fig_irf, axes_irf = plt.subplots(len(WINDOW_LIST), len(IRF_REGIMES), figsize=(16, 12), sharey=True, sharex=True)
    months = np.arange(IRF_HORIZON + 1)

    for i, w in enumerate(WINDOW_LIST):
        for j, (name, end_date_str) in enumerate(IRF_REGIMES.items()):
            end_date = pd.to_datetime(end_date_str)
            start_date = end_date - pd.DateOffset(months=w)
            window_df = df.loc[start_date:end_date]

            res = VAR(window_df).fit(VAR_LAG)
            irf = res.irf(IRF_HORIZON)
            response = irf.orth_irfs[:, idx_fed, idx_ted] 
            lower, upper = irf.errband_mc(orth=True, repl=MC_REPLICATIONS, signif=SIGNIFICANCE, seed=SEED)
            
            ax = axes_irf[i, j]
            ax.plot(months, response, color=COLORS_REGIMES[name], linewidth=2.5)
            ax.fill_between(months, lower[:, idx_fed, idx_ted], upper[:, idx_fed, idx_ted], color=COLORS_REGIMES[name], alpha=0.15)
            ax.axhline(0, color='black', linewidth=1, linestyle='--')
            
            if i == 0:
                ax.set_title(name, fontsize=12, fontweight='bold')
            if j == 0:
                ax.set_ylabel(f"Window: {w}m\nFed Response", fontsize=10, fontweight='bold')
            if i == len(WINDOW_LIST) - 1:
                ax.set_xlabel('Months after shock', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig('09_irf_robustness_test.png', dpi=300, bbox_inches='tight')
    plt.close()

    # B. Multi-window FEVD Panel (36 to 84m)
    fig_fevd, axes_fevd = plt.subplots(len(WINDOW_LIST), 1, figsize=(12, 14), sharex=True)
    date_subprime, date_lehman = pd.to_datetime('2007-08-01'), pd.to_datetime('2008-09-15')

    for j_idx, w in enumerate(WINDOW_LIST):
        dates, rows = [], []
        for i in range(w, len(df)):
            window_df = df.iloc[i - w: i]
            try:
                fevd = VAR(window_df).fit(VAR_LAG).fevd(FEVD_HORIZON)
                d = fevd.decomp[idx_fed, -1, :] 
                dates.append(df.index[i - 1])
                rows.append([d[idx_fed], d[idx_ted], d[idx_un]])
            except:
                continue

        df_fevd = pd.DataFrame(np.array(rows) * 100, index=dates,
                               columns=['Own Shocks (Fed)', 'TED Spread', 'Unemployment'])
        
        ax = axes_fevd[j_idx]
        ax.stackplot(df_fevd.index, df_fevd['Own Shocks (Fed)'], df_fevd['TED Spread'], df_fevd['Unemployment'],
                     labels=['Own Shocks (Fed)', 'TED Spread', 'Unemployment'],
                     colors=['#2ca02c', '#d62728', '#1f77b4'], alpha=0.85)

        ax.axvline(date_subprime, color='orange', linestyle='--', linewidth=1.5)
        ax.axvline(date_lehman, color='black', linestyle=':', linewidth=2.0)
        ax.set_ylabel('Proportion (%)', fontsize=9, fontweight='bold')
        ax.set_title(f"Variance Decomposition - {w}-month Window", fontsize=10, fontweight='bold')
        ax.set_ylim(0, 100)
        
        if j_idx == 0:
            ax.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9, fontsize=8)

    axes_fevd[-1].set_xlabel('Year', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig('09_fevd_robustness_test.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("[SUCCESS] Appendix robustness panels successfully generated.")

# ==========================================
# 4. METRICS & OVERFITTING TESTS
# ==========================================
def calculate_metrics():
    print("\n[*] Calculating Optimization Metrics (RMSFE) and Overfitting Tests...")
    metrics_results = {}

    for w in WINDOW_LIST:
        bics, err_h1, err_h3, unstable, total_windows = [], [], [], 0, 0
        
        for i in range(len(df) - w - FORECAST_HORIZON):
            train, test_h1, test_h3 = df.iloc[i:i+w], df.iloc[i+w], df.iloc[i+w+2]
            try:
                fit = VAR(train).fit(VAR_LAG)
                bics.append(fit.bic)
                if not fit.is_stable(verbose=False): unstable += 1
                total_windows += 1
                
                forecast = fit.forecast(train.values[-VAR_LAG:], steps=FORECAST_HORIZON)
                err_h1.append(np.mean((test_h1.values - forecast[0])**2))
                err_h3.append(np.mean((test_h3.values - forecast[2])**2))
            except:
                continue
                
        metrics_results[w] = {
            'BIC': np.mean(bics), 'RMSFE_h1': np.sqrt(np.mean(err_h1)), 
            'Instability': (unstable / total_windows) * 100 if total_windows > 0 else 0
        }

    print("\n--- Optimization and Overfitting Test Results ---")
    for w, met in metrics_results.items():
        print(f"Window {w}m -> BIC: {met['BIC']:.4f} | RMSFE (h=1): {met['RMSFE_h1']:.4f} | Instability: {met['Instability']:.1f}%")

    mortgage_dates = pd.to_datetime(['2000-01-01', '2008-09-01', '2013-03-01', '2016-12-01', '2026-06-01'])
    duration_months = (mortgage_dates[1:] - mortgage_dates[:-1]).days / 30.437
    avg_crisis = np.mean(duration_months[1:3])
    winning_window = min(WINDOW_LIST, key=lambda x: abs(x - avg_crisis))
    print(f"=> Average duration of acute mortgage crisis cycles: {avg_crisis:.1f} months")
    print(f"=> STRUCTURAL DECISION: The {winning_window}-month window is optimal.")

if __name__ == "__main__":
    generate_section_43_outputs()
    generate_appendix_robustness_panels()
    calculate_metrics()